# 10장 실습 — 배차와 할당 문제

호출 여러 건이 동시에 들어왔을 때 어느 차를 누구에게 보낼지 정하는 문제입니다.
교재 10장에 대응합니다.

작은 비용행렬에서는 모든 배정을 열거해 최솟값을 구할 수 있습니다. 이 값을 기준으로 SciPy 할당 해법의 반환값을 확인합니다.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect, todo
from smartmob.viz import use_korean_font

use_korean_font()

## 1. 비용행렬 (교재 10.1)

행이 승객, 열이 차량입니다. 칸 하나가 "이 차가 이 승객에게 가는 데 걸리는 분"입니다.

In [ ]:
import numpy as np

from smartmob.teaching.dispatch import cost_matrix

passengers = [(37.539, 127.215), (37.545, 127.200), (37.552, 127.190)]
vehicles = [(37.540, 127.210), (37.560, 127.195), (37.535, 127.225)]

costs = cost_matrix(passengers, vehicles)
np.round(costs, 2)

## 2. 탐욕 배차 (교재 10.2)

입력 순서대로 남은 차량 중 비용이 가장 작은 차량을 선택합니다.

In [ ]:
from smartmob.teaching.dispatch import greedy_match, optimal_match

g = greedy_match(costs)
for m in g.matches:
    print(f"승객 {m.passenger} ← 차량 {m.vehicle}  {m.cost:.2f}분")
print(f"합계 {sum(m.cost for m in g.matches):.2f}분")

## 3. 할당 문제 (교재 10.3)

선택한 비용의 합이 가장 작아지도록 배정을 한꺼번에 계산합니다. 이 목적함수에는 요청 순서나 최대 대기시간의 형평성은 포함되지 않습니다.

In [ ]:
o = optimal_match(costs)
for m in o.matches:
    print(f"승객 {m.passenger} ← 차량 {m.vehicle}  {m.cost:.2f}분")
print(f"합계 {sum(m.cost for m in o.matches):.2f}분")

## 4. 최적해가 정말 최적인지 확인합니다

`optimal_match`는 SciPy의 `linear_sum_assignment`를 호출합니다. 현재 SciPy 문서에서는 수정 Jonker–Volgenant 알고리즘을 사용한다고 설명합니다.

승객을 차량에 배정하는 모든 경우를 열거하고 그중 합이 가장 작은 값을 구합니다. 5×5 행렬에는 120가지 배정이 있습니다.

In [ ]:
from itertools import permutations


def brute_force(costs):
    """모든 배정을 세어 보고 합이 가장 작은 것을 돌려줍니다.

    승객 수와 차량 수가 같은 정사각 행렬만 다룹니다.
    n! 가지를 모두 열거하므로 작은 정사각 행렬에만 사용합니다.
    그래서 실전용이 아니라 정답지용입니다.
    """
    n = costs.shape[0]
    best_order, best_total = None, float("inf")
    for order in permutations(range(n)):
        total = sum(costs[i, order[i]] for i in range(n))
        if total < best_total:
            best_order, best_total = order, total
    return best_order, best_total

무작위 행렬 200개를 만들어 둘을 맞춰 봅니다.
3장에서 다익스트라를 NetworkX 와 30쌍 맞춰 본 것과 같은 일입니다.

In [ ]:
rng = np.random.default_rng(42)

banner("SciPy 할당 해법 vs 완전탐색 (5×5, 200회)")
worst = 0.0
mismatch = 0
for _ in range(200):
    m = rng.uniform(1, 30, size=(5, 5))
    _, brute_total = brute_force(m)
    solver_total = sum(x.cost for x in optimal_match(m).matches)
    gap = abs(brute_total - solver_total)
    worst = max(worst, gap)
    mismatch += gap > 1e-9

expect("합이 다른 경우", mismatch, 0)
print(f"    최대 오차 {worst:.2e}")

200개 행렬에서 SciPy 해법과 완전탐색의 총비용이 허용 오차 안에서 일치합니다.

같은 행렬에서 탐욕 배차의 총비용이 최적해보다 얼마나 큰지도 계산합니다.

In [ ]:
banner("탐욕 vs 최적 (5×5, 200회)")
losses = []
for _ in range(200):
    m = rng.uniform(1, 30, size=(5, 5))
    _, best = brute_force(m)
    greedy_total = sum(x.cost for x in greedy_match(m).matches)
    losses.append(greedy_total / best - 1)

print(f"탐욕이 더 쓴 시간  평균 {np.mean(losses):.1%}, 최악 {np.max(losses):.1%}")
print(f"탐욕이 최적과 같았던 비율  {np.mean(np.array(losses) < 1e-9):.0%}")

## 5. 완전탐색을 쓸 수 없는 이유

정답지는 작은 문제에서만 만들 수 있습니다. 경우의 수가 계승으로 늘어나기 때문입니다.

In [ ]:
import math

import pandas as pd

rows = [{"n": n, "경우의 수": math.factorial(n)} for n in [3, 5, 8, 10, 12, 15, 20]]
table = pd.DataFrame(rows)
table["초 (1초에 100만 가지)"] = (table["경우의 수"] / 1e6).round(1)
table

승객과 차량이 각각 20이면 배정 수는 20!, 약 243경입니다. 완전탐색 대신 다항 시간 할당 알고리즘을 사용합니다.

## 6. 비용행렬 크기별 실행시간

In [ ]:
import time

banner("크기별 실행시간")
for n in [5, 20, 100, 300]:
    m = rng.uniform(1, 30, size=(n, n))
    t0 = time.perf_counter()
    optimal_match(m)
    solver = time.perf_counter() - t0

    t0 = time.perf_counter()
    greedy_match(m)
    greedy = time.perf_counter() - t0

    print(f"{n:4d}×{n:<4d}  SciPy 할당 {solver * 1000:7.2f} ms   탐욕 {greedy * 1000:7.2f} ms")

이 실행에서는 100×100과 300×300에서 SciPy 할당 해법이 파이썬 탐욕 구현보다 빠릅니다. 교차 지점은 실행 환경에 따라 달라집니다.

## 7. 픽업 소요시간 계산 방법 (교재 10.6)

지금까지 비용은 직선거리를 평균 속도로 나눈 값이었습니다.
직선거리 비용과 도로망 최단경로 비용이 같은 배정을 만드는지 비교합니다.

In [ ]:
from smartmob.data import load_road_graph
from smartmob.teaching.dispatch import cost_matrix_from_router

G = load_road_graph("hanam", modes=("drive",))

straight = cost_matrix(passengers, vehicles)
road = cost_matrix_from_router(passengers, vehicles, G)

print("직선거리 기준")
print(np.round(straight, 2))
print("\n도로망 기준")
print(np.round(road, 2))

print("\n같은 배정을 고르는가:",
      [m.vehicle for m in optimal_match(straight).matches]
      == [m.vehicle for m in optimal_match(road).matches])

## 8. 배차 조건 바꾸기

### 8.1 탐욕의 처리 순서

`greedy_match` 는 `order` 로 처리 순서를 바꿀 수 있습니다.
순서를 무작위로 바꿔 100번 돌려, 합계가 가장 좋았을 때와 나빴을 때의 차이를 구합니다.
먼저 부른 사람부터 처리하는 것이 좋은 규칙인지 한 줄로 적습니다.

In [ ]:
order_best = None       # 순서를 바꿔 얻은 가장 좋은 합계 (분)
order_worst = None      # 가장 나빴던 합계 (분)

banner("빈칸 8.1")
todo("가장 좋았던 합계", order_best)
todo("가장 나빴던 합계", order_worst)

### 8.2 승객과 차량 수가 다를 때

승객 5명에 차량 3대인 행렬을 만들어 `optimal_match` 를 돌립니다.
배차받지 못한 승객이 `unmatched` 에 들어옵니다.
어떤 승객이 남는지 확인하고, 총비용 최소화 목적에 대기 순서가 포함되는지도 적습니다.

In [ ]:
unmatched_count = None      # 배차받지 못한 승객 수

banner("빈칸 8.2")
todo("배차받지 못한 승객", unmatched_count)

### 8.3 도로망 기준이 답을 바꾸는 경우

승객과 차량 위치를 바꿔 가며 직선거리 기준과 도로망 기준이 다른 배정을 내는 경우를 하나 찾고, 두 비용행렬을 함께 출력합니다.

In [ ]:
found_case = None       # 배정이 갈린 (passengers, vehicles) 한 쌍

banner("빈칸 8.3")
todo("배정이 갈린 사례", found_case, fmt=lambda x: "찾았습니다")

## 정리

- 탐욕 배차는 입력 순서대로 차량을 고르고, 할당 해법은 전체 비용의 합을 최소화합니다
- 작은 문제에서는 완전탐색 결과로 SciPy 할당 해법을 검증할 수 있습니다
- 완전탐색의 경우의 수는 계승으로 증가하므로 큰 문제에는 사용할 수 없습니다
- 구현별 실행시간은 같은 크기의 비용행렬로 측정합니다
- 직선거리와 도로망 최단경로 비용은 서로 다른 배정을 만들 수 있습니다
- 11장 실습에서는 이 배차를 1분마다 부르는 루프를 직접 짭니다